In [1]:
#| include: false
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pyarrow as pa
from pathlib import Path
import ipywidgets as widgets
import plotly.io as pio
pio.renderers.default = "notebook_connected"


get_ipython().ast_node_interactivity = "all"

In [2]:
#| include: false

CSV_PATH = Path.cwd().parent / "output" / "counts" / "amz_job_counts_by_category_2022.csv"
counts = pd.read_csv(CSV_PATH, parse_dates=["date"])
counts.shape
counts.head()
counts.columns = ["job_category", "count", "date"]
# counts.to_parquet(Path("./output/counts/amz_job_counts_by_category_2022.parquet"), engine="pyarrow", index=False)
# counts.to_csv(Path("../output/counts/amz_job_counts_by_category_2022.csv"), index=False)


(8262, 3)

,job_category,count,date
0,Administrative Support,666.0,2022-06-19
1,Amazon Design,1372.0,2022-06-19
2,Audio / Video / Photography Production,208.0,2022-06-19
3,Business & Merchant Development,2283.0,2022-06-19
4,Business Intelligence,2439.0,2022-06-19


In [3]:
#| include: false

'''Target Insights:

1. Read the data: csv at first. Save as parquet for future use.
2. How many days of coverage are there?
3. How many unique job categories are there?
4. Have the jobs changed over time? (i.e. are there new job categories that have appeared, or old ones that have disappeared?)
5. What was the start date and end date of the data?
6. Were any days missed in the period? Identify them
7. What was the maximum number of positions for a given job category? Start with Data Science in the first instance.
8. What was the minimum number of positions for a given job category? Start with Data Science in the first instance.
9. Identify the percentage number of open positions for a given job category from the daily total number of open positions. Start with Data Science in the first instance.
10. Check for seasonality in the data. Are there any patterns that can be identified on a weekday, weekly or monthly basis? Start with Data Science in the first instance.
11. Are the job counts normally distributed? Start with Data Science in the first instance.
12. Are there any significant changes in the job counts over time? Start with Data Science in the first instance.
13. Are there any outlier values in the job counts? Start with Data Science in the first instance.
14. If there are significant changes in the job counts, can they be explained by external events? Start with Data Science in the first instance.
15. Are there any correlations between the job counts and other external factors? Start with Data Science in the first instance.
16. Are there any correlations between the job counts and other job categories? Start with Data Science in the first instance.
17. Does Amazon discriminate between job categories or advertise different job categories on different sites?

'''

'Target Insights:\n\n1. Read the data: csv at first. Save as parquet for future use.\n2. How many days of coverage are there?\n3. How many unique job categories are there?\n4. Have the jobs changed over time? (i.e. are there new job categories that have appeared, or old ones that have disappeared?)\n5. What was the start date and end date of the data?\n6. Were any days missed in the period? Identify them\n7. What was the maximum number of positions for a given job category? Start with Data Science in the first instance.\n8. What was the minimum number of positions for a given job category? Start with Data Science in the first instance.\n9. Identify the percentage number of open positions for a given job category from the daily total number of open positions. Start with Data Science in the first instance.\n10. Check for seasonality in the data. Are there any patterns that can be identified on a weekday, weekly or monthly basis? Start with Data Science in the first instance.\n11. Are the

In [4]:
#| include: false

# How many days of coverage are there?
print(f'# of days of coverage: {counts.date.nunique()}')
print(f'start date: {counts.date.min()}')
print(f'end date: {counts.date.max()}') 
print(f'# calendar days in period: {(counts.date.max() - counts.date.min()).days + 1}')
print(f'150 days were missed in the period.')
max_counts = counts[counts["job_category"] == "Data Science"]["count"].max()
print(f'# max number of Data Science positions: {max_counts}')
print(f'Date of max number of Data Science positions: {counts[(counts["count"] == max_counts) & (counts["job_category"] == "Data Science")]["date"].values[0]}')
min_counts = counts[counts["job_category"] == "Data Science"]["count"].min()
print(f'# min number of Data Science positions: {min_counts}')
print(f'Date of min number of Data Science positions: {counts[(counts["count"] == min_counts) & (counts["job_category"] == "Data Science")]["date"].values[0]}')

print(f'\n{display(counts.head())}')


# of days of coverage: 243
start date: 2022-06-19 00:00:00
end date: 2023-07-16 00:00:00
# calendar days in period: 393
150 days were missed in the period.
# max number of Data Science positions: 767.0
Date of max number of Data Science positions: 2022-06-19T00:00:00.000000
# min number of Data Science positions: 7.0
Date of min number of Data Science positions: 2023-01-14T00:00:00.000000


,job_category,count,date
0,Administrative Support,666.0,2022-06-19
1,Amazon Design,1372.0,2022-06-19
2,Audio / Video / Photography Production,208.0,2022-06-19
3,Business & Merchant Development,2283.0,2022-06-19
4,Business Intelligence,2439.0,2022-06-19



None


In [5]:
#| include: false

## Examine missing dates in the data. The data should be daily, but there are some missing days.
expected_dates = pd.date_range(counts["date"].min(), counts["date"].max(), freq="D")
missing_dates = expected_dates.difference(counts["date"].drop_duplicates())

print(f"Missing dates: {len(missing_dates)}")
missing_dates
missing_dates.min()
missing_dates.max()

Missing dates: 150


DatetimeIndex(['2022-11-06', '2022-11-07', '2022-11-08', '2022-11-30',
               '2022-12-01', '2023-01-05', '2023-01-07', '2023-01-08',
               '2023-01-09', '2023-01-11',
               ...
               '2023-07-04', '2023-07-05', '2023-07-06', '2023-07-07',
               '2023-07-09', '2023-07-10', '2023-07-11', '2023-07-13',
               '2023-07-14', '2023-07-15'],
              dtype='datetime64[us]', length=150, freq=None)

Timestamp('2022-11-06 00:00:00')

Timestamp('2023-07-15 00:00:00')

In [6]:
#| include: false

missing_dates_red = missing_dates[missing_dates >= pd.Timestamp("2022-11-09")]
missing_dates_red.min()
missing_dates_red.max()

Timestamp('2022-11-30 00:00:00')

Timestamp('2023-07-15 00:00:00')

## What story am I trying to tell?

In 2022, I decided to collect data on the number of Amazon job openings, by job category, on [Amazon's jobs' website](https://amazon.jobs/content/en/job-categories). 
The Amazon job openings were advertised under 34 different job categories during the period, such as Administrative Support, Business Intelligence, Data Science, … etc. I was motivated to observe these counts because I saw Amazon as a bellwether company for the tech industry. 

I collected this data on 243 days between the dates of June 19, 2022 and July 16, 2023, a span of 393 calendar days.

The results were startling. The total number of Amazon job openings fell by 95% during the period and have not really recovered since. What follows are summaries of the observations and potential explanations for the dramatic changes in the data.


## Overview of the Amazon job categories and counts

The Amazon job categories, the number of job openings per category and the total number of job openings at the beginning and at the end of the period are shown in the following table. The table also shows the percentage change in the number of job openings over the periord. 

In [7]:
#| include: false

## Create a table showing the number of jobs by category at the beginning and end of the period.

def get_job_counts_by_category(counts, date):
    return counts[counts["date"] == date][["job_category", "count"]].sort_values(by="count", ascending=False).reset_index(drop=True).astype({"count": int})

initial_jobs = get_job_counts_by_category(counts, counts["date"].min())
final_jobs = get_job_counts_by_category(counts, counts["date"].max())

def calculate_total_jobs_and_add_row_to_table(counts):
    total_jobs = counts["count"].sum()
    total_row = pd.DataFrame({"job_category": ["Total"], "count": [total_jobs]})
    return pd.concat([counts, total_row], ignore_index=True)

def combine_job_counts(initial, final):
    combined = initial.merge(final, on="job_category", how="outer", suffixes=("_initial", "_final"))
    combined["count_initial"] = combined["count_initial"].fillna(0).astype(int)
    combined["count_final"] = combined["count_final"].fillna(0).astype(int)
    return combined.sort_values(by="count_initial", ascending=False).reset_index(drop=True)


def calculate_difference(combined):
    combined["difference"] = combined["count_final"] - combined["count_initial"]
    return combined

def calculate_difference_percentage(combined):
    combined["difference_percentage"] = round((combined["difference"] / combined["count_initial"].replace(0, 1) * 100), 1)
    return combined

# print(display(calculate_total_jobs_and_add_row_to_table(initial_jobs)))
# print(display(calculate_total_jobs_and_add_row_to_table(final_jobs)))

combined_table = combine_job_counts(initial_jobs, final_jobs)
# combined_table = combine_job_counts(calculate_total_jobs_and_add_row_to_table(initial_jobs), calculate_total_jobs_and_add_row_to_table(final_jobs))
combined_table = calculate_difference(combined_table)
combined_table = calculate_difference_percentage(combined_table)
combined_table.columns = ["Job Category", "June 2022 Count", "July 2023 Count", "Change", "% Change"]
print(display(combined_table))




,Job Category,June 2022 Count,July 2023 Count,Change,% Change
0,Software Development,32437,635,-31802,-98.0
1,Solutions Architect,6627,162,-6465,-97.6
2,"Operations, IT, & Support Engineering",5453,909,-4544,-83.3
3,Project/Program/Product Management--Technical,5281,295,-4986,-94.4
4,Project/Program/Product Management--Non-Tech,5140,489,-4651,-90.5
5,"Sales, Advertising, & Account Management",4431,569,-3862,-87.2
6,Human Resources,3767,221,-3546,-94.1
7,"Systems, Quality, & Security Engineering",3242,418,-2824,-87.1
8,Finance and Global Business Services,2909,351,-2558,-87.9
9,Business Intelligence,2439,113,-2326,-95.4


None


The job categories with the largest decreases in open positions were, in order: Software Development (31,802), Solutions Architect (6,465), Project/Program/Product Management combined (9,673) and Operations, IT, & Support Engineering (4,544). 

In percentage terms, Software Development (98%), Public Relations & Communications (98%), Solutions Architect (98%) and Data Science (97%) showed the steepest declines. The overall percentage decline in the number of open positions was 91%.

The job categories with the smallest percentage decline in open positions included: Medical, Health & Safety (23%), Fulfillment & Operations Management (32%), Facilities Maintenance & Real Estate (56%) and Investigation & Loss Prevention (68%).


In [8]:
#| include: false
print(display(combined_table))

,Job Category,June 2022 Count,July 2023 Count,Change,% Change
0,Software Development,32437,635,-31802,-98.0
1,Solutions Architect,6627,162,-6465,-97.6
2,"Operations, IT, & Support Engineering",5453,909,-4544,-83.3
3,Project/Program/Product Management--Technical,5281,295,-4986,-94.4
4,Project/Program/Product Management--Non-Tech,5140,489,-4651,-90.5
5,"Sales, Advertising, & Account Management",4431,569,-3862,-87.2
6,Human Resources,3767,221,-3546,-94.1
7,"Systems, Quality, & Security Engineering",3242,418,-2824,-87.1
8,Finance and Global Business Services,2909,351,-2558,-87.9
9,Business Intelligence,2439,113,-2326,-95.4


None


In [9]:
#| include: false

# combined_table.sort_values("Change", ascending=True)
# combined_table.sort_values("Change", ascending=False)
combined_table.sort_values("% Change", ascending=False)


,Job Category,June 2022 Count,July 2023 Count,Change,% Change
33,Fulfillment Associate,55,48,-7,-12.7
22,"Medical, Health, & Safety",704,543,-161,-22.9
13,Fulfillment & Operations Management,1565,1065,-500,-31.9
18,"Facilities, Maintenance, & Real Estate",977,433,-544,-55.7
24,Investigation & Loss Prevention,531,169,-362,-68.2
21,Supply Chain/Transportation Management,726,228,-498,-68.6
17,Customer Service,1028,251,-777,-75.6
19,Hardware Development,877,208,-669,-76.3
16,"Buying, Planning, & Instock Management",1058,201,-857,-81.0
23,Administrative Support,666,125,-541,-81.2


In [10]:
#| echo: false

import dash
from dash import Dash, dash_table, html

def build_interactive_table_app(df: pd.DataFrame) -> Dash:
    """Build a Dash app showing a DataFrame as a filterable, sortable table."""
    app = Dash(__name__)
    app.layout = html.Div(
        [
            html.H3("Amazon Job Counts by Category"),
            dash_table.DataTable(
                id="combined-table",
                data=df.to_dict("records"),
                columns=[
                    {
                        "name": col,
                        "id": col,
                        "type": "numeric" if pd.api.types.is_numeric_dtype(df[col]) else "text",
                    }
                    for col in df.columns
                ],
                filter_action="native",
                sort_action="native",
                page_size=34,
                style_table={"overflowX": "auto"},
                style_cell={"textAlign": "left", "padding": "4px"},
                style_header={"fontWeight": "bold"},
            ),
        ],
        style={"margin": "20px"},
    )
    return app

dash_app = build_interactive_table_app(combined_table)
dash_app.run(jupyter_mode="inline", debug=False)

## Change in Amazon Open Positions by Job Category

The following graphs show the change in open job positions during the evaluation period for the selected job category. You can select the job category to inspect by clicking on the job category pull down menu on the top left of the graph.

### Software Developer
The decline in open positions for Software Developer is stark. The number of positions remained persistently low once its minimum was reached and was 98% below the opening count at the end of the period.

In [11]:
#| echo: false

import plotly.graph_objects as go

DEFAULT_CATEGORY = "Software Development"


def build_category_figure(df: pd.DataFrame, job_category: str) -> go.Figure:
    """Build a line chart of open positions over time for one job category."""
    category_data = df.loc[df["job_category"].eq(job_category)].sort_values("date")
    legend_label = (
        f"Count of Amazon open positions for: {job_category}; "
        "June 2022 - July 2023"
    )
    fig = go.Figure(
        go.Scatter(
            x=category_data["date"],
            y=category_data["count"],
            mode="lines+markers",
            name=legend_label,
        )
    )
    fig.update_layout(
        xaxis_title="Date",
        yaxis_title="# Open positions",
        showlegend=True,
        legend={"orientation": "h", "yanchor": "bottom", "y": 1.04, "x": 0},
        template="plotly_white",
    )
    return fig


def build_category_dropdown_figure(
    df: pd.DataFrame, default_category: str = DEFAULT_CATEGORY
) -> go.Figure:
    """Build one figure with a trace per category and a dropdown to switch between them."""
    categories = sorted(df["job_category"].dropna().unique())
    fig = go.Figure()
    for category in categories:
        trace = build_category_figure(df, category).data[0]
        trace.visible = category == default_category
        fig.add_trace(trace)

    buttons = [
        {
            "label": category,
            "method": "update",
            "args": [{"visible": [c == category for c in categories]}],
        }
        for category in categories
    ]
    fig.update_layout(
        title="Amazon Open Positions by Job Category",
        xaxis_title="Date",
        yaxis_title="# Open positions",
        showlegend=True,
        legend={"orientation": "h", "yanchor": "bottom", "y": 0.97, "x": 0},
        template="plotly_white",
        margin={"t": 140},
        updatemenus=[
            {
                "buttons": buttons,
                "active": categories.index(default_category),
                "direction": "down",
                "x": 0,
                "xanchor": "left",
                "y": 1.25,
                "yanchor": "top",
            }
        ],
    )
    return fig


build_category_dropdown_figure(counts)

### Data Science
The decline in open positions for Data Science is also severe. It tracks closely with the decline in Software Developer positions in the period.

In [12]:
#| echo: false
build_category_figure(counts, "Data Science")

### Public Relations & Communications
Public Relations & Communications is one of the categories for which the observed declines were very dramatic. The largest declines appear to be concentrated in the period Nov 2022 - January 2023. Retrospective analysis indicates that Amazon instituted deliberate, hiring restrictions / embargoes in November 2022 and in January 2023 and that these restrictions are likely the cause of the near step-function decreases in open positions.  

In [13]:
#| echo: false
build_category_figure(counts, "Public Relations & Communications")

### Fulfillment & Operations Management
Unlike the Software Developer and Data Science categories, Fulfillment & Operations Management showed evidence of a rebound after the minimum was reached in January 2023. Intuitively, this makes some sense, as Amazon still has to fulfill its orders for its retail business to maintain revenues, even if it is reducing costs elsewhere in the business. 

In [14]:
#| echo: false
build_category_figure(counts, "Fulfillment & Operations Management")

### Medical, Health, & Safety
Medical, Health, & Safety showed a particularly strong rebound following the January 2023 minimum and ended the period 23% below its opening count. This decline is much lower than the 91% decline in total open positions during the period. Overall, it seems the tech and project management positions were targeted for the brunt of the cutbacks.

In [15]:
#| echo: false
build_category_figure(counts, "Medical, Health, & Safety")